# 05 HRV Domain And Robustness

Reviewer-facing HRV-domain and robustness readiness/status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This branch does not yet include implemented HRV-domain or robustness stress-test runners. This notebook validates upstream evidence, writes blocked/readiness ledgers, and prevents accidental placeholder execution. It does not train models, perturb signals, or create robustness metrics without implemented scripts.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

RUN_HEAVY_HRV_ROBUSTNESS = False

revision_root = Path('reports/revision')
freeze_path = revision_root / 'manifests' / 'oof_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / 'calibration_ci_oof_full_predictions.json'
baseline_summary_path = revision_root / 'metrics' / 'baseline_summary.csv'
component_summary_path = revision_root / 'metrics' / 'component_check_summary.json'
audit_path = revision_root / 'audit_protocol.json'
a0_status_path = revision_root / 'a0_resolution_status.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'

required_inputs = {
    'oof_freeze_manifest': freeze_path,
    'calibration_ci': calibration_path,
    'baseline_summary': baseline_summary_path,
    'component_check_summary': component_summary_path,
    'audit_protocol': audit_path,
    'a0_resolution_status': a0_status_path,
    'hrv36_schema': hrv_schema_path,
}
for name, path in required_inputs.items():
    print(f'{name:26s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 01/02/03/04 artifacts are required before notebook 05: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('OOF freeze manifest must be status=frozen and manuscript_ready=true before HRV/robustness review.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'allowed_execution': {
        'hrv_domain_runner': RUN_HEAVY_HRV_ROBUSTNESS,
        'robustness_stress_runner': RUN_HEAVY_HRV_ROBUSTNESS,
        'model_training': False,
        'model_inference': False,
    },
}
save_json(revision_root / 'manifests' / 'hrv_robustness_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'hrv_robustness_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 05 to HRV-domain and robustness tasks in the tracked revision plan.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_HRV', 'EXP_ROBUST'])]
relevant_claims = claims[claims['claim_id'].isin(['C03', 'C04'])]
relevant_tasks = tasks[tasks['id'].isin(['A6', 'B2'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json')


## HRV Domain Evidence Ledger

The required HRV-only and HRV-domain analyses are not implemented yet. This table records the exact blocker and prevents the current OOF result from being used as HRV-domain evidence.


In [ ]:
hrv_schema = pd.read_csv(hrv_schema_path)
print('HRV schema rows:', len(hrv_schema))
display(hrv_schema.head(10))

hrv_rows = [
    {
        'analysis_name': 'HRV-only baseline',
        'required_output': 'HRV-only metrics under same split/threshold protocol',
        'status': 'blocked_runner_tbd',
        'metric_name': 'roc_auc_macro',
        'metric_value': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented HRV-only baseline runner in scripts/revision.',
        'safe_wording': 'Do not claim HRV branch contribution from current artifacts.',
    },
    {
        'analysis_name': 'HRV domain classifier',
        'required_output': 'Domain AUC and interpretation of HRV dataset-source sensitivity',
        'status': 'blocked_runner_tbd',
        'metric_name': 'domain_auc',
        'metric_value': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented HRV domain classifier runner in scripts/revision.',
        'safe_wording': 'Do not claim HRV domain robustness; state domain-bias check remains pending.',
    },
    {
        'analysis_name': 'Duration/noise HRV sensitivity',
        'required_output': 'Duration/noise sensitivity summary for HRV features',
        'status': 'blocked_runner_tbd',
        'metric_name': 'sensitivity_delta',
        'metric_value': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented duration/noise HRV sensitivity runner in scripts/revision.',
        'safe_wording': 'Do not present HRV descriptors as validated against duration/noise shifts.',
    },
]
hrv_df = pd.DataFrame(hrv_rows)
hrv_csv = revision_root / 'metrics' / 'hrv_domain_summary.csv'
hrv_table = revision_root / 'tables' / 'table_hrv_domain_status.csv'
hrv_df.to_csv(hrv_csv, index=False)
hrv_df.to_csv(hrv_table, index=False)
print('Wrote:', hrv_csv)
print('Wrote:', hrv_table)
display(hrv_df)


## Robustness Stress-Test Ledger

The revised plan requires perturbation tests. Since no runner exists in this branch, the notebook writes a blocked but reusable robustness table instead of fabricating metrics.


In [ ]:
robustness_rows = []
for analysis_name, perturbation in [
    ('SNR 20 dB noise', 'additive_noise_snr_20db'),
    ('SNR 10 dB noise', 'additive_noise_snr_10db'),
    ('SNR 5 dB noise', 'additive_noise_snr_5db'),
    ('Random 3-lead dropout', 'random_3_lead_dropout'),
    ('V1-V6 dropout', 'precordial_v1_v6_dropout'),
    ('500Hz to 250Hz sampling shift', 'sampling_rate_500_to_250hz'),
]:
    robustness_rows.append({
        'analysis_name': analysis_name,
        'perturbation': perturbation,
        'dataset': 'chapman_oof',
        'status': 'blocked_runner_tbd',
        'clean_metric_reference': str(calibration_path),
        'perturbed_metric_value': math.nan,
        'absolute_delta': math.nan,
        'relative_delta': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented robustness stress-test runner in scripts/revision.',
        'safe_wording': 'Do not claim robustness under this perturbation until the runner is implemented and artifacts are frozen.',
    })

robustness_df = pd.DataFrame(robustness_rows)
robustness_csv = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_table = revision_root / 'tables' / 'table_robustness.csv'
robustness_df.to_csv(robustness_csv, index=False)
robustness_df.to_csv(robustness_table, index=False)
print('Wrote:', robustness_csv)
print('Wrote:', robustness_table)
display(robustness_df)


## Claim Evidence Summary

This JSON is the rebuttal-facing interpretation boundary for HRV and robustness claims.


In [ ]:
summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'hrv_schema': {'path': str(hrv_schema_path), 'sha256': sha256_file(hrv_schema_path)},
    'hrv_status': 'blocked_runner_tbd',
    'robustness_status': 'blocked_runner_tbd',
    'claim_boundaries': [
        {
            'claim_id': 'C03',
            'claim_topic': 'HRV provides complementary rhythm descriptors',
            'current_status': 'not_supported_by_current_artifacts',
            'safe_wording': 'HRV schema is documented, but HRV-only/domain evidence remains pending.',
        },
        {
            'claim_id': 'A6/R2C3',
            'claim_topic': 'minimum robustness stress tests',
            'current_status': 'not_supported_by_current_artifacts',
            'safe_wording': 'Robustness tests must be implemented and reported as absolute/relative degradation before any robustness claim.',
        },
    ],
}
summary_path = revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json'
save_json(summary_path, summary)
print('Wrote:', summary_path)


## Heavy Runner Guard

This cell records the intended command surface. It fails closed if heavy execution is enabled before scripts exist.


In [ ]:
planned = {
    'HRV-only baseline': 'python scripts/revision/TBD_hrv_only.py',
    'HRV domain classifier': 'python scripts/revision/TBD_hrv_domain_classifier.py',
    'Duration/noise HRV sensitivity': 'python scripts/revision/TBD_hrv_duration_noise_sensitivity.py',
    'Robustness stress tests': 'python scripts/revision/TBD_robustness_stress.py',
}

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:32s}: implemented={script.exists()} planned_command={command}')

if RUN_HEAVY_HRV_ROBUSTNESS:
    missing = [name for name, command in planned.items() if not Path(command.split()[1]).exists()]
    if missing:
        raise RuntimeError('HRV/robustness runners are not implemented yet: ' + ', '.join(missing))
    for name, command in planned.items():
        run(command, log_path=f'reports/revision/logs/{name.lower().replace("/", "_").replace(" ", "_")}.log')
else:
    print('Heavy HRV/robustness execution disabled. Current notebook only writes traceable blocked-status ledgers.')


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_robustness_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_robustness_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'hrv_robustness_input_contract.json',
    revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'metrics' / 'robustness_summary.csv',
    revision_root / 'tables' / 'table_robustness.csv',
    revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 05 complete. HRV-domain and robustness claims remain blocked until dedicated runners are implemented and frozen.')
